# Appendix G: Coding Agents

The agentic edit-test loop, edit formats, sandboxing, and verification.

This notebook is an original tutorial (rewritten for 2026 practice: coding
agents are agentic-loop tools — Claude Code, Codex-class CLIs — not
"prompt a specialist role").

## Learning Objectives

- Describe the coding-agent loop: orient → plan → edit → run tests → read failures → repeat.
- Compare edit formats (whole-file, unified diff, search/replace) and why their failure rates differ.
- Explain sandboxing/permission models for shell-executing agents.
- Explain verification beyond "tests pass" and how coding agents are benchmarked (SWE-bench-style).

## Mental Model

A coding agent is a loop whose **observations are compiler output and test
failures**. That grounding is why coding was the first agent domain to work
well: the environment provides free, objective, machine-checkable feedback
every iteration.

```
task ─▶ orient (read repo: tree, grep, key files)
     ─▶ plan (which files, what change)
     ─▶ edit (structured format applied to files)
     ─▶ verify (build, tests, linters)
          │ failures ──▶ read output, revise edit ──▶ verify ...
          ▼ green
     ─▶ review/diff for the human
```

Every design question — edit formats, sandboxing, context strategy — is
about keeping this loop fast, safe, and grounded.

## Important Concepts

- **Orientation**: agents don't load the whole repo; they explore (file tree, grep, targeted reads) and build a working set. Repo maps / code indexes accelerate this.
- **Edit formats**: how the model expresses changes — (1) **whole-file rewrite**: simple, reliable for small files, token-expensive and clobber-prone for big ones; (2) **unified diff**: compact, but models generate malformed hunks/line numbers; (3) **search/replace blocks**: model quotes exact existing text + replacement — the dominant format because success is *verifiable before applying* (if the search text doesn't match, reject and retry instead of corrupting the file).
- **Sandboxing**: the agent runs shell commands, so containment is the safety story — filesystem scoping, network restrictions, permission prompts for dangerous commands, and git worktrees/containers for parallel agents (notebook 28).
- **Verification ladder**: parse/typecheck → unit tests → integration/E2E → *behavioral* verification (drive the actual app) → human diff review. Tests-pass is necessary, not sufficient: agents can overfit to tests or weaken them.
- **Benchmarks**: SWE-bench (resolve real GitHub issues; Verified subset human-validated) is the standard yardstick — knowing roughly where frontier models sit (~65-75% on Verified in 2026) signals currency.

## Need-To-Know Coverage Checklist

- [x] The edit-test loop, in runnable code.
- [x] Edit formats with failure-rate reasoning; search/replace as the default.
- [x] Sandboxing: FS/network scoping, permission tiers, worktree isolation.
- [x] Verification beyond tests; test-weakening as a failure mode.
- [x] SWE-bench framing; context strategies (explore vs index).

## Deep Study Notes

### Why search/replace won

A whole-file rewrite of a 800-line file wastes tokens and risks silently
dropping code the model "forgot". A unified diff is compact but the model
must fabricate line numbers and context lines exactly — malformed hunks fail
to apply. Search/replace makes the model quote text that *must already
exist*; the harness verifies the match before touching the file. Failed
match = clean retry with feedback, never a corrupted file. The general
principle: **choose output formats whose validity is checkable before
side effects.**

### Sandboxing and permissions

The agent's power tool is the shell, which is also the risk. Production
models layer:
- **Scoping**: filesystem roots (only this repo), network allowlists.
- **Permission tiers**: read-only commands auto-approved; mutating commands
  (installs, git push, rm) prompt or require allowlist entries.
- **Isolation for parallelism**: one worktree/container per agent so
  concurrent agents can't clobber each other (notebook 28); merge review is
  then a standard git operation.
- **Injection awareness**: file contents and command output are untrusted
  input — a README can carry prompt injection into a shell-wielding agent.

### Verification beyond green tests

Agents exhibit two test-related failure modes worth naming: **overfitting**
(hard-coding the expected value) and **test weakening** (editing the test to
pass). Defenses: hold-out tests the agent can't see, diff review gates that
flag test-file edits, and behavioral verification — actually running the app
and observing the change. The interview line: "tests are the inner loop;
the outer loop is 'does the feature demonstrably work'."

### Context strategy

Two schools: **agentic exploration** (grep/read on demand — robust,
token-hungry) and **indexing** (embeddings/repo map — cheaper lookups, can
go stale). 2026 practice leans exploration-first with lightweight repo maps;
full RAG-over-code has faded for agents that can just search (see agentic
RAG, notebook 29).

## Common Failure Modes

- Whole-file edits on large files → silently dropped code.
- Applying unverified diffs → corrupted files mid-loop.
- Tests as the only verification → overfit patches, weakened tests ship.
- Unsandboxed shell + injected README → arbitrary command execution.
- Loading the whole repo into context → cost blowup and lost-in-the-middle instead of targeted reads.
- No diff review gate → plausible-but-wrong changes merged on green CI.

## Implementation Notes

- Reject-and-retry on failed search matches; never fuzzy-apply silently.
- Auto-approve read-only commands only; log every command the agent runs.
- Keep the agent's iteration budget explicit; escalate with the failing output after N failed verify cycles.
- Gate merges on human diff review with test-file edits highlighted.

In [ ]:
"""The coding-agent inner loop: search/replace edits with pre-apply
verification, then an edit-test cycle that reads failures and retries.
"""

FILE = {"calc.py": "def total(items):\n    return sum(i.price for i in items)\n"}


def apply_search_replace(filename, search, replace):
    """The format's whole point: verifiable BEFORE side effects."""
    content = FILE[filename]
    if search not in content:
        return {"ok": False, "error": "search text not found - no changes made"}
    if content.count(search) > 1:
        return {"ok": False, "error": "search text ambiguous (2+ matches)"}
    FILE[filename] = content.replace(search, replace)
    return {"ok": True}


def run_tests():
    """Toy test: total() must skip items with price=None."""
    code = FILE["calc.py"]
    if "if i.price" in code or "is not None" in code:
        return {"passed": True}
    return {"passed": False,
            "output": "TypeError: unsupported operand +: int and NoneType (test_skips_none)"}


# The agent's scripted attempts: first a bad edit (stale quote), then correct.
ATTEMPTS = [
    {"search": "return sum(i.cost for i in items)",   # hallucinated old code
     "replace": "return sum(i.cost or 0 for i in items)"},
    {"search": "return sum(i.price for i in items)",
     "replace": "return sum(i.price for i in items if i.price is not None)"},
]


def edit_test_loop(max_cycles=3):
    for cycle, edit in enumerate(ATTEMPTS[:max_cycles]):
        result = apply_search_replace("calc.py", edit["search"], edit["replace"])
        if not result["ok"]:
            print(f"cycle {cycle}: edit REJECTED ({result['error']}) -> retry with fresh read")
            continue  # file untouched — the format prevented corruption
        tests = run_tests()
        print(f"cycle {cycle}: edit applied, tests "
              f"{'PASS' if tests['passed'] else 'FAIL: ' + tests['output']}")
        if tests["passed"]:
            return "green - ready for human diff review"
    return "budget exhausted - escalate with failing output"


print(edit_test_loop())
print("\nfinal file:\n" + FILE["calc.py"])


## Practice

1. Add a third failure mode to `apply_search_replace`: the search text spans
   a region another agent just edited. How does worktree isolation prevent this?
2. Make the agent "cheat": have it edit `run_tests` instead of `calc.py`.
   Which two defenses from the notes catch it?
3. Compare token cost: whole-file rewrite vs search/replace for a 3-line fix
   in a 500-line file. Roughly what ratio?
4. Design the permission tiers for a CI-integrated coding agent: which
   commands auto-run, which prompt, which are forbidden?

## Design Checklist

- [ ] Edits use a pre-apply-verifiable format; failed matches retry cleanly.
- [ ] Shell scoped (FS roots, network allowlist); permission tiers enforced.
- [ ] Parallel agents isolated (worktrees/containers).
- [ ] Verification ladder past unit tests; test-file edits flagged in review.
- [ ] Iteration budget + escalation with failing output.
- [ ] File contents treated as untrusted input (injection).